# 04a. 로지스틱 회귀 경량 모델

- 담당: **④ 경량화·통합** (이 파일은 담당자만 수정)
- 목적: 11/06 개인 실험(리더보드 0.2854)을 팀 공통 함수로 재현하고, ③의 그룹 K-fold와 ② 피처로 점수 비교
- 실험 (모두 C=0.1, class_weight=balanced, 그룹 K-fold)
  - 0~5: 0/1 + 변이 개수 → `1109_logreg_C01_gkf`
  - 6~7: + 변이 종류 5개(①) → `1109_logreg_C01_derived_gkf`
- 기록 파일: `experiments/log_P_light.csv`
- 예측 파일: `experiments/preds/P_light__logreg_C01_1109__*`, `P_light__logreg_C01_derived_1109__*`
- ⚠️ Run All 하면 "5. 팀 기록표 기록"과 "7. 기록 + 예측 파일 저장"이 다시 실행돼 기록표에 같은 줄이 또 생겨요.** 다시 돌릴 때는 기록 줄을 정리하거나 실험 이름을 바꿔 주세요.

> **규칙:** 전처리는 Pipeline 안에 / test는 보지 않기 / 점수는 `run_cv()`로 / 결과는 `log_result()`로 기록 / 자세한 규칙은 `docs/coding_rules.md`

## 0. 환경 설정 (수정하지 말고 그대로 실행)
Colab은 구글 드라이브 `내 드라이브`에 레포 폴더(`ah07-cancer-subtype`, 안에 `data/` 포함)를 두면 돼요.

In [1]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/ah07-cancer-subtype"  # 드라이브에 둔 레포 폴더
else:
    REPO_DIR = os.path.abspath(os.path.join("..", ".."))  # notebooks/이니셜_역할/ 폴더 기준

# 경로 확인: 여기서 멈추면 폴더 위치나 이름을 확인해 주세요
assert os.path.isdir(os.path.join(REPO_DIR, "src")), (
    f"src 폴더를 찾을 수 없어요: {REPO_DIR}\n"
    "→ 드라이브 '내 드라이브' 바로 아래에 'ah07-cancer-subtype' 폴더가 있는지, "
    "ZIP을 풀 때 폴더가 두 번 겹치지 않았는지(ah07-cancer-subtype/ah07-cancer-subtype-dev) 확인")
assert os.path.isfile(os.path.join(REPO_DIR, "data", "train.csv")), (
    "data/train.csv 가 없어요 → 데이콘 csv 3개를 data/ 폴더에 넣어 주세요")
sys.path.insert(0, REPO_DIR)

from src.config import SEED, DATA_DIR, set_seed
set_seed(SEED)
print("REPO_DIR:", REPO_DIR, "| DATA_DIR:", DATA_DIR)

REPO_DIR: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype | DATA_DIR: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype/data


## 1. 도구 불러오기
일반 도구(numpy·pandas·scikit-learn)와 팀 공통 함수(`src` 폴더)를 불러오기.

In [2]:
# ── 일반 도구 ──────────────────────────────────────────
import numpy as np                                    # 숫자 계산
import pandas as pd                                   # 표(데이터프레임) 다루기
from scipy import sparse                              # 0이 대부분인 데이터를 가볍게 저장하는 형식
from sklearn.base import clone                        # 모델 설정을 그대로 복사 (새로 학습할 때)
from sklearn.pipeline import Pipeline                 # "피처 만들기 → 모델" 순서를 한 묶음으로
from sklearn.preprocessing import FunctionTransformer # 내가 만든 함수를 Pipeline에 넣을 수 있게 포장
from sklearn.linear_model import LogisticRegression   # 경량 모델: 로지스틱 회귀

# ── 팀 공통 코드 (src 폴더) ─────────────────────────────
from src.data_loader import load_data, split_xy, encode_target    # 데이터 읽기, 문제/정답 나누기, 암종 이름→숫자
from src.preprocessing import binarize_mutations                  # 팀 기본 피처: WT·빈칸=0, 변이=1
from src.validation import get_folds, run_cv                      # 팀 공통 5등분, 교차검증 채점
from src.evaluation import per_class_f1, log_result, save_predictions  # 암종별 점수, 기록표 적기, 답안지 저장

print("✅불러오기 완료")

✅불러오기 완료


## 2. 데이터 불러오기
train을 문제(유전자 변이)와 정답(암종)으로 나누고, 팀 공통 5-fold를 생성. test는 마지막 예측에만 사용.

In [ ]:
# 데이터 읽기 (DATA_DIR은 환경 설정에서 정해짐 → VS Code·Colab 모두 자동)
train, test = load_data(DATA_DIR)      # test는 마지막 예측에만 사용 (학습·분석에 쓰지 않음)

# train을 문제(X)와 정답(y)으로 나누기
X, y_raw = split_xy(train)             # X: 유전자 4,384개 열 / y_raw: 암종 이름(BRCA 등)
y, le = encode_target(y_raw)           # 암종 이름을 0~25 숫자로 (le로 다시 이름으로 되돌림)

# 팀 전원이 똑같이 쓰는 5등분 방법
folds = get_folds(y)

print("train", train.shape, "| test", test.shape, "| 암종 수", len(le.classes_))

train (6201, 4386) | test (2546, 4385) | 암종 수 26


## 3. 피처 생성 — 유전자 변이 여부(0/1) + 환자별 변이 개수
팀 기본 피처(WT·빈칸=0, 변이=1)에 환자별 변이 개수 log(1+개수)를 1열 추가. 한 환자 안에서만 계산해 누수 없음.

In [ ]:
def binary_plus_count(X):
    """
    피처 = 유전자별 변이 여부(0/1) + 환자별 변이 개수
    한 환자(한 행) 안에서만 계산 → test 정보를 쓰지 않으므로 누수 없음
    """
    b = binarize_mutations(X).astype(np.float32)       # 팀 기본 피처: WT·빈칸=0, 변이=1 (11/06 제출 코드와 같은 숫자 형식)
    cnt = np.log1p(b.sum(axis=1).to_numpy())[:, None]  # 환자별 변이 개수 → log(1+개수)로 크기 줄이기
    return sparse.hstack([                             # 두 가지를 옆으로 붙이기
        sparse.csr_matrix(b.to_numpy(dtype=np.float32)),   # 0/1 표 (4,384열)
        sparse.csr_matrix(cnt)                             # 변이 개수 (1열)
    ]).tocsr()

# Pipeline에 넣을 수 있게 포장
light_features = FunctionTransformer(binary_plus_count)
print("피처 함수 준비 완료")

피처 함수 준비 완료


## 4. 교차검증 — 로지스틱 회귀 (C=0.1, class_weight=balanced)
팀 공통 `run_cv()`로 5-fold Macro F1을 계산.

In [ ]:
# 모델 설정 (11/06 제출 모델과 같은 값, 숫자로 기록 → 대회 재현 규정)
LR_PARAMS = dict(
    C=0.1,                     # 규제 세기: 작을수록 단순한 모델 (C=1보다 0.1이 좋았음)
    max_iter=2000,             # 학습 반복 최대 횟수
    class_weight="balanced",   # 환자 수가 적은 암종을 틀리면 더 크게 벌점 → Macro F1에 유리
)

# 피처 만들기 → 모델 학습을 한 묶음으로
pipe_lr = Pipeline([
    ("features", light_features),
    ("model", LogisticRegression(**LR_PARAMS)),
])

# 팀 공통 채점: 5등분해서 4개로 학습, 1개로 시험 × 5번 → 평균 점수
cv_lr = run_cv(pipe_lr, X, y, folds=folds)

fold 1: Macro F1 = 0.4159
fold 2: Macro F1 = 0.4157
fold 3: Macro F1 = 0.3808
fold 4: Macro F1 = 0.4103
fold 5: Macro F1 = 0.4220
CV Macro F1 = 0.4089 (std 0.0145)


## 5. 팀 기록표 기록 + 예측 파일 저장
`log_result()`로 기록표에 한 줄 남기고, train 전체로 다시 학습한 뒤 `save_predictions()`로 앙상블·제출용 예측 파일 저장.

In [ ]:
# ① 팀 기록표에 한 줄 추가 (experiments/log_P_light.csv)
log_result(
    owner="P_light",                          # 담당 (내 폴더 이름)
    notebook="04a_logreg_baseline.ipynb",     # 이 노트북 파일 이름
    experiment="1109_logreg_C01_gkf",             # 실험 이름 (날짜_모델_설정)
    features="binary + log1p(n_mutated)",     # 사용한 피처
    model="LogisticRegression",               # 사용한 모델
    params=LR_PARAMS,                         # 모델 설정값
    cv=cv_lr,                                 # 교차검증 결과 (평균·편차 자동 기록)
    notes="경량 베이스라인, 그룹 K-fold, 리더보드 0.2854", # 메모
)

# ② train 전체로 다시 학습 (교차검증은 채점용, 제출용은 전체 데이터로)
final_lr = clone(pipe_lr).fit(X, y)

# ③ test 예측 확률 (26개 암종별 확률)
test_proba = final_lr.predict_proba(test[X.columns])

# ④ 답안지 저장 (experiments/preds/) → 앙상블·제출 후보가 됨
save_predictions(
    owner="P_light",
    name="logreg_C01_1109",                   # 파일 이름에 들어갈 이름 (밑줄 두 개 __ 금지)
    oof_proba=cv_lr["oof_proba"],             # 교차검증 중 시험 본 부분의 예측 확률
    test_proba=test_proba,                    # test 예측 확률
    classes=le.classes_,                      # 암종 이름 순서
    cv=cv_lr,
    notes="경량 베이스라인",
)

기록: log_P_light.csv ← 1109_logreg_C01_gkf (CV 0.4089)
예측 저장: P_light__logreg_C01_1109 (CV 0.4089) → experiments/preds/


## 6. 같은 피처 비교 — 로지스틱 회귀 + ③ 최신 피처 (그룹 K-fold)
③ 최신 XGBoost와 같은 피처(0/1 + 변이 개수 + 변이 유형별 개수)로 로지스틱 회귀를 돌려 모델 차이만 비교해요.

In [7]:
from src.features import mutation_count, mutation_type_counts

def derived_features_lr(X):
    """③ 최신 피처(0/1 + 변이 개수 + 변이 유형별 개수). 개수는 log(1+개수)로 크기를 줄임."""
    X_filled = X.fillna("WT")
    b = binarize_mutations(X_filled).astype(np.float32)
    counts = pd.concat([mutation_count(X_filled), mutation_type_counts(X_filled)], axis=1)
    counts = np.log1p(counts.to_numpy(dtype=np.float32))
    return sparse.hstack([sparse.csr_matrix(b.to_numpy()),
                          sparse.csr_matrix(counts)]).tocsr()

pipe_lr_derived = Pipeline([
    ("features", FunctionTransformer(derived_features_lr)),
    ("model", LogisticRegression(**LR_PARAMS)),
])
cv_lr_derived = run_cv(pipe_lr_derived, X, y, folds=folds)

fold 1: Macro F1 = 0.4238
fold 2: Macro F1 = 0.4187
fold 3: Macro F1 = 0.3881
fold 4: Macro F1 = 0.4338
fold 5: Macro F1 = 0.4460
CV Macro F1 = 0.4221 (std 0.0194)


## 7. 기록 + 예측 파일 저장 — 로지스틱 회귀 + ③ 최신 피처
CV가 0/1 + 변이 개수 모델(0.4089)과 ③ 최신 XGBoost(0.4183)보다 높아 기록표와 예측 파일로 남겨요.

In [8]:
log_result(
    owner="P_light",
    notebook="04a_logreg_baseline.ipynb",
    experiment="1109_logreg_C01_derived_gkf",
    features="binary + log1p(n_mutated, 변이 유형별 개수 5개)",
    model="LogisticRegression",
    params=LR_PARAMS,
    cv=cv_lr_derived,
    notes="③ 최신 XGBoost와 같은 피처, 그룹 K-fold",
)

final_lr_derived = clone(pipe_lr_derived).fit(X, y)
test_proba_derived = final_lr_derived.predict_proba(test[X.columns])

save_predictions(
    owner="P_light",
    name="logreg_C01_derived_1109",
    oof_proba=cv_lr_derived["oof_proba"],
    test_proba=test_proba_derived,
    classes=le.classes_,
    cv=cv_lr_derived,
    notes="로지스틱 회귀 + ③ 최신 피처, 그룹 K-fold",
)

기록: log_P_light.csv ← 1109_logreg_C01_derived_gkf (CV 0.4221)
예측 저장: P_light__logreg_C01_derived_1109 (CV 0.4221) → experiments/preds/
